# Lectures 7-15: SDK Foundations in Colab

This notebook combines the Section 3 hands-on lessons into one Colab-ready workflow:

- Lecture 7: Installing and running the SDK
- Lecture 8: Validating and explaining standards files
- Lecture 9: Using ODPV vocabulary helpers
- Lecture 10: Working with local LLMs
- Lecture 11: Working with online LLM providers
- Lecture 12: Generating ODPS data products
- Lecture 13: Generating ODPC fragments
- Lecture 14: Working with ODPG graphs
- Lecture 15: Working with ODPC catalogs

Run the cells from top to bottom. Some generation cells require an `ANTHROPIC_API_KEY` stored in Colab secrets. The validation, vocabulary, graph, and catalog cells can still run without an LLM key.


### What We Just Covered

Before opening the hands-on workbook, we introduced the SDK as the practical tooling layer for the Open Data Products standards family. The SDK helps turn standards from static documents into workflows for validation, generation, explanation, catalogs, graphs, HTML review pages, and agent-ready YAML.

In this notebook, we now move from that mental model into the first executable SDK exercises.

# Lecture 7: Installing and Running the SDK

This section prepares Colab, clones the course exercise repository, installs the SDK, and creates one workspace folder for Section 3 outputs.


### What We Just Covered

We compared the main setup options: local terminal, Python notebook, and Google Colab. The important idea is that the SDK workflow is the same even when the runtime changes: install the package, check the command works, work inside the cloned lesson folders, and keep outputs organized.

Next, we install the SDK and run the first checks in Colab.

## Clone The Course Exercises

The course repository contains the sample files used by the lessons. In this workbook, we run the exercises directly inside the cloned lecture folders under:

```text
/content/odps-sdk-course-exercises/
```

Generated files are written beside each lesson's sample files in folders such as `products/`, `fragments/`, or `output/`.

In [ ]:
%cd /content
!rm -rf /content/odps-sdk-course-exercises
!git clone https://github.com/Open-Data-Product-Initiative/odps-sdk-course-exercises.git
%cd /content/odps-sdk-course-exercises

## Install The SDK

Install the latest released SDK into the Colab runtime. Re-run this cell if Colab restarts.


In [ ]:
!python -m pip install --upgrade open-data-products==0.2.3


## Check The SDK Command

The SDK CLI is named `open-data-products`.


In [ ]:
!open-data-products --version
!open-data-products --help


## Run A Simple Machine-Readable Command

`manifest --json` proves the package imports and the CLI can render structured output. JSON is most useful for scripts, automation, and agents.


### Before You Run This

Many SDK commands can produce either human-readable output or structured JSON. In this quick check, `--json` shows the SDK as an automation tool: scripts and AI agents can read the same command output that a person can inspect in the notebook.

In [ ]:
!open-data-products manifest --json | python -m json.tool | head -60


# Lecture 8: Validating and Explaining Standards Files

Validation checks whether YAML follows an Open Data Product family standard. Explanation gives a compact human-readable view of the artifact.


### What We Just Covered

Validation is the trust layer. Before we generate or combine artifacts, we need to know whether a standards file has the required structure and fields. A validation error is not just a failure; it is useful feedback that helps keep bad artifacts out of catalogs, graphs, and automation.

Next, we create a valid and an intentionally invalid product file and compare the SDK feedback.

## Create Valid And Invalid ODPS Files


### Before You Run This

This lesson uses two small YAML files from the cloned course repository. One is valid, and one is intentionally schema-invalid. The invalid file is still valid YAML text; it fails because it is missing required ODPS content. That distinction matters: syntax errors and standards validation errors teach different things.

In [ ]:
%cd /content/odps-sdk-course-exercises/08-validating-and-explaining-standards-files
!find . -maxdepth 1 -type f | sort

## Validate A Correct Product


In [ ]:
%cd /content/odps-sdk-course-exercises/08-validating-and-explaining-standards-files
!open-data-products validate product.yaml
!open-data-products explain product.yaml

## Validate A Schema-Invalid Product

This file is readable YAML, but it is missing the required `productID` field.


### Before You Run This

Expect this command to report a validation problem. That is the point of the exercise. In real work, this feedback tells you what to fix before the artifact enters a catalog, graph, or automated workflow.

In [ ]:
!open-data-products validate invalid-product.yaml --json


## Load A Lightweight Summary

`summary` returns file-level metadata and references. It does not return the full document body.


In [ ]:
!open-data-products summary product.yaml --json | python -m json.tool


# Lecture 9: Use The ODPV Vocabulary Helpers

ODPV helpers make shared vocabulary easier to search, resolve, explain, and check.


### What We Just Covered

ODPV helps keep language consistent across product specs, catalogs, and generated outputs. Without controlled vocabulary, the same idea can appear under many labels, which makes review and automation harder.

Next, we use vocabulary helpers to summarize, search, resolve, and inspect vocabulary relationships.

## Summarize And Search The Vocabulary


### Before You Run This

Vocabulary commands are not generating new data products. They help you inspect the controlled language available in ODPV so that product specs, catalogs, and generated fragments use consistent terms.

In [ ]:
!open-data-products odpv-summary --json | python -m json.tool | head -80
!open-data-products odpv-search "governance policy risk" --limit 3 --json | python -m json.tool


## Resolve, Explain, And Check Relationships


In [ ]:
!open-data-products odpv-resolve "reusable data asset" --json | python -m json.tool
!open-data-products odpv-explain DataProduct --json | python -m json.tool
!open-data-products odpv-relationship DataProduct supports UseCase --json | python -m json.tool


# Lecture 10: Working With Local LLMs

Local LLMs are useful for cost control, development, privacy, and restricted environments. In the main guide this lesson uses Ollama locally.

Colab is not the best place to run that part because the notebook runtime does not normally have your local Ollama server. Treat this lecture as a local-machine workflow, then use the online-provider cells below for Colab execution.


### What We Just Covered

Local LLMs can help with cost control, privacy, restricted environments, and repeatable development tests. The tradeoff is that local setup depends on the learner's machine and model runtime, so this Colab workbook treats local execution as a pattern to understand rather than a required cloud exercise.

Next, we review the local-machine workflow before moving to online provider generation.

## Local Machine Pattern

The local LLM lesson has sample files in the cloned course repo under:

```text
/content/odps-sdk-course-exercises/10-working-with-local-llms/
```

When running outside Colab, open the same lesson folder in your local clone and run the commands from inside it:

```bash
ollama pull qwen2.5
ollama list
open-data-products generate \
  --config generation.config.yaml \
  --input source_docs/turnaround-delay-signal.txt \
  --kind signal
```

The important SDK idea is the same in local and online modes: source text goes in, standards-shaped YAML comes out.

# Lecture 11: Working With Online LLM Providers

Colab works best with hosted providers. Store your API key in Colab secrets as `ANTHROPIC_API_KEY`, then load it into the notebook environment.


### What We Just Covered

Online providers let the same SDK workflow run against hosted models. The key separation is configuration versus secrets: config selects provider and model, while API keys stay in environment variables or notebook secrets.

Next, we store the provider key safely, create a generation config, and run an online generation example.

## Store The Provider Key

Do not paste real API keys into notebook cells. Use Colab secrets instead.


### Before You Run This

Hosted LLM providers need API keys. The safe pattern is to keep secrets outside YAML config files and source files. In Colab, use notebook secrets; locally, use environment variables.

In [ ]:
import os

try:
    from google.colab import userdata
    key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    key = os.environ.get("ANTHROPIC_API_KEY")

if key:
    os.environ["ANTHROPIC_API_KEY"] = key
    print("ANTHROPIC_API_KEY is available for this runtime.")
else:
    print("ANTHROPIC_API_KEY is not set. LLM generation cells will be skipped until you add the secret.")


## Create A Generation Config

The config keeps provider details separate from the command you run.


In [ ]:
%cd /content/odps-sdk-course-exercises/11-working-with-online-llm-providers
!rm -rf fragments
!mkdir -p fragments
!find . -maxdepth 2 -type f | sort

## Generate One Product Reference Fragment

This command uses the source file and writes one ODPC product reference fragment.


### Before You Run This

A product reference fragment is a small portfolio object, not a full ODPS product specification. It is useful when you want a catalog-ready reference that can later be combined with objectives, use cases, signals, and graph relationships.

In [ ]:
%cd /content/odps-sdk-course-exercises/11-working-with-online-llm-providers
if os.environ.get("ANTHROPIC_API_KEY"):
    !open-data-products generate --config generation.config.yaml --kind product-reference
    !find fragments -maxdepth 1 -type f | sort
else:
    print("Skipped: add ANTHROPIC_API_KEY to Colab secrets to run this generation cell.")

# Lecture 12: Generating ODPS Data Products From Business Requirements

This lecture generates full ODPS product YAML drafts from source documents. These are data product drafts, not catalog fragments.


### What We Just Covered

Business requirements can become structured ODPS data product drafts, but generated YAML is still a draft. The useful loop is generate, inspect, improve the source or prompt if needed, and validate.

Next, we prepare source documents and generate minimal ODPS product drafts.

## Prepare Source Documents

Copy the course sample documents into the Section 3 workspace.


In [ ]:
%cd /content/odps-sdk-course-exercises/12-generating-odps-data-products
!rm -rf products
!mkdir -p products
!find source_docs -maxdepth 1 -type f | sort

## Generate Minimal ODPS Product Drafts

The `minimal` profile stays close to source-backed facts.


### Before You Run This

Here the target is different: `--kind odps-product` asks for full data product drafts. The `minimal` profile keeps the generated files smaller so they are easier to inspect first. Generated YAML should always be reviewed and validated before reuse.

In [ ]:
%cd /content/odps-sdk-course-exercises/12-generating-odps-data-products
if os.environ.get("ANTHROPIC_API_KEY"):
    !open-data-products generate \
      --provider claude \
      --model claude-sonnet-4-5 \
      --input source_docs/ \
      --kind odps-product \
      --profile minimal \
      --output products/
    !find products -maxdepth 1 -type f | sort
else:
    print("Skipped: add ANTHROPIC_API_KEY to Colab secrets to run ODPS generation.")

## Optional Complete Draft Profile

Use `complete-draft` when you want the SDK to draft common review-needed ODPS components such as SLA, data quality, and pricing plans.


In [ ]:
# Optional. This can use more tokens than the minimal profile.
# Uncomment after you have reviewed the minimal output.
#
# !open-data-products generate \
#   --provider claude \
#   --model claude-sonnet-4-5 \
#   --input source_docs/ \
#   --kind odps-product \
#   --profile complete-draft \
#   --output products/


## Validate Generated Products


In [ ]:
%cd /content/odps-sdk-course-exercises/12-generating-odps-data-products
!for product_file in products/*.yaml; do \
  if [ -f "$product_file" ]; then open-data-products validate "$product_file"; fi; \
done

# Lecture 13: Working With Generation And Fragments

Fragments are smaller ODPC authoring units. They are useful when you want independently reviewable products, use cases, objectives, and signals before building a catalog or graph.


### What We Just Covered

Fragments are small standalone portfolio objects. They are easier to review than one large artifact and can later be assembled into catalogs and graphs. This is the bridge from one generated product draft toward portfolio-level work.

Next, we generate ODPC fragments and build a catalog from them.

## Prepare Fragment Source Folders


In [ ]:
%cd /content/odps-sdk-course-exercises/13-working-with-generation-and-fragments
!rm -rf fragments catalog.yaml
!mkdir -p fragments
!find source_docs -type f | sort

## Generate ODPC Fragments

Each command uses one source lane and one fragment kind.


### Before You Run This

This step generates several fragment types from separate source folders. The goal is not one big file yet; it is a set of smaller objects that can be reviewed independently and then assembled into a catalog.

In [ ]:
%cd /content/odps-sdk-course-exercises/13-working-with-generation-and-fragments
if os.environ.get("ANTHROPIC_API_KEY"):
    !open-data-products generate --provider claude --model claude-sonnet-4-5 --input source_docs/products/ --kind product-reference --output fragments/
    !open-data-products generate --provider claude --model claude-sonnet-4-5 --input source_docs/use-cases/ --kind use-case --output fragments/
    !open-data-products generate --provider claude --model claude-sonnet-4-5 --input source_docs/objectives/ --kind objective --output fragments/
    !open-data-products generate --provider claude --model claude-sonnet-4-5 --input source_docs/signals/ --kind signal --output fragments/
    !find fragments -maxdepth 1 -type f | sort
else:
    print("Skipped: add ANTHROPIC_API_KEY to Colab secrets to generate fragments.")

## Build A Catalog From Generated Fragments

Run this after the generation cell has produced fragment YAML files.


### Before You Run This

The catalog build step collects separate fragments into one ODPC catalog. Think of fragments as the pieces and the catalog as the organized portfolio view of those pieces.

In [ ]:
%cd /content/odps-sdk-course-exercises/13-working-with-generation-and-fragments
if len([p for p in __import__('pathlib').Path('fragments').glob('*.yaml')]) > 0:
    !open-data-products odpc-build fragments/ --output catalog.yaml
    !open-data-products validate catalog.yaml
else:
    print("Skipped: no generated fragment YAML files found yet.")

# Lecture 14: Working With ODPG Graphs

ODPG graphs make relationships between products, use cases, objectives, and signals explicit. This part uses prepared sample fragments, so it can run even if you skipped LLM generation.


### What We Just Covered

Catalogs describe portfolio objects; graphs describe relationships between those objects. In graph terms, products, use cases, objectives, and signals become nodes, while links between them become edges.

Next, we build, validate, render, and package graph output.

## Copy Sample Fragments


In [ ]:
%cd /content/odps-sdk-course-exercises/14-working-with-odpg-graphs
!rm -rf output
!mkdir -p output
!find fragments -maxdepth 1 -type f | sort

## Build, Validate, And Render The Graph


### Before You Run This

The graph is about relationships. Products, use cases, objectives, and signals become nodes; their links become edges. The rendered HTML helps humans inspect those relationships, while `graph.yaml` stays machine-readable.

In [ ]:
%cd /content/odps-sdk-course-exercises/14-working-with-odpg-graphs
!open-data-products odpg-build fragments/ \
  --output output/graph.yaml \
  --id customer-retention-graph \
  --name "Customer Retention Graph"
!open-data-products validate output/graph.yaml
!open-data-products odpg-summary output/graph.yaml
!open-data-products odpg-generate output/graph.yaml --output output/graph-explorer.html

## Extract Agent Context Around One Node


### Before You Run This

Agent context is a focused slice of the graph around one node. Instead of giving an AI agent the whole portfolio, this command retrieves the nearby relationships that matter for a specific product, use case, or objective.

In [ ]:
!open-data-products odpg-agent-context output/graph.yaml \
  --node PR-CUSTOMER-HEALTH-SIGNALS \
  --depth 2 \
  --json | python -m json.tool


## Package The Graph Output

Download the zip if you want to inspect the graph explorer locally.


In [ ]:
!zip -r odpg-graph-output.zip output


# Lecture 15: Working With ODPC Catalogs

ODPC catalogs collect product references, use cases, business objectives, and signals into one portfolio artifact.


### What We Just Covered

ODPC catalogs combine portfolio objects into a structured portfolio artifact. Fragments are the authoring units, the catalog is the combined structure, and the graph explains relationships.

Next, we build and render a catalog, then close Section 3 before moving to the portfolio builder workflow.

## Copy Sample Fragments


In [ ]:
%cd /content/odps-sdk-course-exercises/15-working-with-odpc-catalogs
!rm -rf output
!mkdir -p output
!find fragments -maxdepth 1 -type f | sort

## Build And Render The Catalog


### Before You Run This

This repeats the catalog idea with ready-made fragments. The YAML file is the structured artifact; the HTML file is for human review. Both views come from the same portfolio objects.

In [ ]:
%cd /content/odps-sdk-course-exercises/15-working-with-odpc-catalogs
!open-data-products odpc-build fragments/ \
  --output output/catalog.yaml \
  --html output/catalog.html
!open-data-products validate output/catalog.yaml
!open-data-products odpc-summary output/catalog.yaml

## Search ODPC Guidance

This searches bundled ODPC object guidance, not the generated catalog file.


In [ ]:
!open-data-products odpc-search "business operational analytical policy user needs" --limit 5


## Package The Catalog Output

Download the zip if you want to inspect the catalog HTML locally.


In [ ]:
!zip -r odpc-catalog-output.zip output


# Section 3 Recap

You have now used the SDK to:

- install and inspect the CLI
- validate, explain, and summarize standards files
- use ODPV vocabulary helpers
- understand where local LLMs fit
- configure a hosted LLM provider in Colab
- generate ODPS product drafts
- generate ODPC fragments
- build and render ODPG graphs
- build and render ODPC catalogs

Section 4 builds on these pieces by turning source material into a complete portfolio workspace.


### What We Just Covered

Section 3 moved from SDK setup into the core building blocks: validation, vocabulary helpers, provider configuration, generation, fragments, graphs, and catalogs. These are the individual capabilities that the final portfolio workflow will combine.

Before continuing, this is a good point to pause for the Section 3 quiz or recap.